In [ ]:
import os
import sys
# Add the parent directory to the Python path to allow module imports
sys.path.append(os.path.dirname(os.getcwd()))

import random
import numpy as np
import pandas as pd
import seaborn as sns
from pprint import pprint
from pathlib import Path
from matplotlib import pyplot as plt

import torch
from torchinfo import summary
from lightning import Trainer
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

from sklearn.preprocessing import label_binarize
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

from modules.datamodule import LungSoundDataModule
from modules.classifier import LungSoundClassifier
from modules.transforms import *

In [ ]:
PROJECT_ROOT = Path(os.path.dirname(os.getcwd()))
DATA_FOLDER = PROJECT_ROOT / "data" / "preprocessed"
if not os.path.exists(DATA_FOLDER):
    print(f"Data directory not found at {DATA_FOLDER}. Please ensure the data is downloaded and placed in the correct location.")

LOGS_FOLDER = PROJECT_ROOT / "logs"
if not os.path.exists(LOGS_FOLDER):
    os.makedirs(LOGS_FOLDER)

CHECKPOINTS_FOLDER = PROJECT_ROOT / "checkpoints"
if not os.path.exists(CHECKPOINTS_FOLDER):
    os.makedirs(CHECKPOINTS_FOLDER)

RESULTS_FOLDER = PROJECT_ROOT / "results"
if not os.path.exists(RESULTS_FOLDER):
    os.makedirs(RESULTS_FOLDER)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

torch.set_float32_matmul_precision("high")
%load_ext tensorboard

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_input_size(dm: LungSoundDataModule) -> tuple:
    """
    Get the input size for the model based on a sample batch from the DataModule.
    Args:
        dm (LungSoundDataModule): The DataModule to sample from.
    Returns:
        input_size (tuple): Tuple representing the shape of the input data (e.g., (B, C, H, W)).
    """
    dm.setup()
    loader = dm.train_dataloader()
    batch = next(iter(loader))
    inputs, _, _ = batch
    return tuple(inputs.shape)


def save_results(model) -> None:
    """
    Save test results to a CSV file.
    Args:
        test_results (dict): Dictionary containing test metrics and results.
        path (Path | str): Path to the CSV file where results will be saved.
    """
    classes = model.hparams.dataset["classes"]
    probs_df = pd.DataFrame(model.test_results["probs"], columns=[f"prob_{cls}" for cls in classes])
    results_df = pd.DataFrame({
        "target": model.test_results["targets"],
        "pred": model.test_results["preds"],
    })
    info_df = pd.DataFrame(model.test_results["info"])
    results_df = pd.concat([results_df, probs_df, info_df], axis=1)
    path = RESULTS_FOLDER / f"{model.hparams['experiment']}_results.csv"
    results_df.to_csv(path, index=False)
    print(f"Test results saved to {path}")


def train_model(hparams: dict, debug: bool = False) -> LungSoundClassifier:
    """
    Train a model.
    Args:
        hparams (dict): Hyperparameters and configuration for the experiment. Trainer parameters:
        ```
        - experiment (str): Name of the experiment for logging and checkpointing.
        - checkpoint_monitor (str): Metric to monitor for checkpointing (e.g., "val_loss").
        - checkpoint_mode (str): Mode for checkpointing ("min" or "max").
        - early_stop_monitor (str): Metric to monitor for early stopping (e.g., "val_loss").
        - early_stop_mode (str): Mode for early stopping ("min" or "max").
        - patience (int): Number of epochs with no improvement after which training will be stopped.
        - max_epochs (int): Maximum number of training epochs.
        - precision (int): Precision for training (e.g., 16 for mixed precision).
        ```
        debug (bool): If True, dry run the training for quick debugging. Default is False.
    Returns:
        model (LungSoundClassifier): The trained model.
    """
    set_seed(hparams["seed"])

    # Display experiment hyperparameters
    print("Experiment hyperparameters:")
    pprint(hparams)

    # Initialize logger
    logger = TensorBoardLogger(
        save_dir=LOGS_FOLDER,
        name=hparams["experiment"],
    )

    # Initialize callbacks
    callbacks = []
    if not debug:
        checkpoint_callback = ModelCheckpoint(
            dirpath=CHECKPOINTS_FOLDER,
            filename=hparams["experiment"] + "-{epoch}-{val_loss:.2f}-{val_f1_micro:.2f}-{val_f1_macro:.2f}",
            monitor=hparams["checkpoint_monitor"],
            mode=hparams["checkpoint_mode"],
            save_top_k=1,
            verbose=True,
        )
        callbacks.append(checkpoint_callback)

        early_stopping_callback = EarlyStopping(
            monitor=hparams["early_stop_monitor"],
            mode=hparams["early_stop_mode"],
            patience=hparams["patience"],
            verbose=True,
        )
        callbacks.append(early_stopping_callback)

    # Initialize DataModule and Model
    dm = LungSoundDataModule(hparams, DATA_FOLDER)
    model = LungSoundClassifier(hparams)

    # Display model architecture
    input_shape = get_input_size(dm)
    print("Model Summary:\n")
    print(f"Input shape: {input_shape}")
    print(summary(model, input_size=input_shape))
    print("\n")

    # Initialize Trainer
    trainer = Trainer(
        accelerator="gpu" if DEVICE == "cuda" else "cpu",
        devices="auto",
        max_epochs=hparams["max_epochs"],
        precision=hparams["precision"],
        callbacks=callbacks,
        logger=logger,
        fast_dev_run=debug,
    )

    # Train the model
    trainer.fit(model, dm)

    # Test the model
    trainer.test(model, dm)

    # Save test results to a CSV
    save_results(model)

    return model

In [ ]:
def show_metrics(targets: np.ndarray, preds: np.ndarray, probs: np.ndarray, class_names: list):
    """
    Display classification metrics including classification report, ROC AUC curve, Precision-Recall curve, and confusion matrix.
    Args:
        targets (np.ndarray): Array of true target labels.
        preds (np.ndarray): Array of predicted labels by the model.
        probs (np.ndarray): Array of predicted probabilities for each class.
        class_names (list): List of class names corresponding to the target labels.
    """
    # y_true in one-hot encoding format for multi-class metrics
    y_true = label_binarize(targets, classes=np.arange(len(class_names)))

    # Print classification report
    print("Classification Report:")
    print(classification_report(targets, preds, target_names=class_names))

    # Plot ROC curves
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, class_name in enumerate(class_names):
        RocCurveDisplay.from_predictions(y_true[:, i], probs[:, i], name=class_name, ax=ax)
    ax.set_title("ROC Curves")
    plt.show()

    # Plot Precision-Recall curves
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, class_name in enumerate(class_names):
        PrecisionRecallDisplay.from_predictions(y_true[:, i], probs[:, i], name=class_name, ax=ax)
    ax.set_title("Precision-Recall Curves")
    plt.show()

    # Plot confusion matrix
    cm = confusion_matrix(targets, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
    plt.title("Confusion Matrix")
    plt.show()


def evaluate_model(model: LungSoundClassifier, test_model: bool = False):
    """
    Evaluate the model on the test set and display metrics.
    Args:
        model (LungSoundClassifier): The trained model to evaluate.
        test_model (bool): If True, run the test loop to populate test results.
    """
    hparams = model.hparams
    if not hasattr(model, "test_results") or test_model:
        dm = LungSoundDataModule(model.hparams, DATA_FOLDER)
        trainer = Trainer(
            accelerator="gpu" if DEVICE == "cuda" else "cpu",
            devices="auto",
            precision=model.hparams["precision"],
            logger=False,
            enable_checkpointing=False,
            enable_progress_bar=False,
            enable_model_summary=False,
        )
        trainer.test(model, dm)

    targets = np.asarray(model.test_results["targets"])
    preds = np.asarray(model.test_results["preds"])
    probs = np.asarray(model.test_results["probs"])
    class_names = hparams.dataset["classes"]
    show_metrics(targets, preds, probs, class_names)

## Experiments with DenseNet121

In [ ]:
hparams = {
    "experiment": "",               # This will be modified for each experiment

    # Callbacks
    "checkpoint_monitor": "val_f1_macro",
    "checkpoint_mode": "max",
    "early_stop_monitor": "val_f1_macro",
    "early_stop_mode": "max",
    "patience": 10,

    # Training
    "max_epochs": 100,
    "precision": "16-mixed",
    "seed": 42,

    # DataModule
    "batch_size": 32,               # TODO: Adjust batch size based on GPU memory
    "num_workers": 8,               # TODO: Adjust number of workers based on CPU cores
    "sampler": "equalizer",

    # Dataset
    "dataset": {
        "name": "Combined_ICBHI_KAUH",
        "classes": [
            "Asthma",
            "Bronchiectasis",
            "Bronchiolitis",
            "COPD",
            "Healthy",
            "Lung Fibrosis",
            "Pneumonia",
            "URTI",
        ],
        "feature_extractor": "",    # This will be modified for each experiment
        "sample_limit": 1000,
        "transform": None,
    },

    # Model
    "model": {
        "name": "densenet121",
        "pretrained": None,
        "freeze_layers": False,
    },

    # Optimization
    "criterion": "CrossEntropyLoss",
    "optimizer": "AdamW",
    "learning_rate": 0.0001,
    "weight_decay": 0.00001,
    "momentum": None,
    "lr_scheduler": "StepLR",
    "step_size": 7,
    "gamma": 0.1,
}

### MagSTFT

In [ ]:
hparams["experiment"] = "densenet121-classes=8-equalizer-sample_limit=1000-MagSTFT"
hparams["dataset"]["feature_extractor"] = "MagSTFT"
model = train_model(hparams)

In [ ]:
evaluate_model(model)

### MelSpectrogram

In [ ]:
hparams["experiment"] = "densenet121-classes=8-equalizer-sample_limit=1000-MelSpectrogram"
hparams["dataset"]["feature_extractor"] = "MelSpectrogram"
model = train_model(hparams)

In [ ]:
evaluate_model(model)

### MFCC

In [ ]:
hparams["experiment"] = "densenet121-classes=8-equalizer-sample_limit=1000-MFCC"
hparams["dataset"]["feature_extractor"] = "MFCC"
model = train_model(hparams)

In [ ]:
evaluate_model(model)

### MFCCDelta

In [ ]:
hparams["experiment"] = "densenet121-classes=8-equalizer-sample_limit=1000-MFCCDelta"
hparams["dataset"]["feature_extractor"] = "MFCCDelta"
model = train_model(hparams)

In [ ]:
evaluate_model(model)

### Chroma

In [ ]:
hparams["experiment"] = "densenet121-classes=8-equalizer-sample_limit=1000-Chroma"
hparams["dataset"]["feature_extractor"] = "Chroma"
model = train_model(hparams)

In [ ]:
evaluate_model(model)

### Phase

In [ ]:
hparams["experiment"] = "densenet121-classes=8-equalizer-sample_limit=1000-Phase"
hparams["dataset"]["feature_extractor"] = "Phase"
model = train_model(hparams)

In [ ]:
evaluate_model(model)

### ImagSTFT and RealSTFT
The combination is a stack of ImagSTFT and RealSTFT features, resulting in a 3-channel feature.

In [ ]:
hparams["experiment"] = "densenet121-classes=8-equalizer-sample_limit=1000-Stack_ImagSTFT_RealSTFT"
hparams["dataset"]["feature_extractor"] = ["ImagSTFT", "RealSTFT"]
model = train_model(hparams)

In [ ]:
evaluate_model(model)

### MagSTFT and Phase
The combination is a stack of MagSTFT and Phase features, resulting in a 3-channel feature.

In [ ]:
hparams["experiment"] = "densenet121-classes=8-equalizer-sample_limit=1000-Stack_MagSTFT_Phase"
hparams["dataset"]["feature_extractor"] = ["MagSTFT", "Phase"]
model = train_model(hparams)

In [ ]:
evaluate_model(model)

### MelSpectrogram, MFCC and Chroma
The combination is a stack of MelSpectrogram, MFCC and Chroma features, resulting in a 3-channel feature.

In [ ]:
hparams["experiment"] = "densenet121-classes=8-equalizer-sample_limit=1000-Stack_MelSpectrogram_MFCC_Chroma"
hparams["dataset"]["feature_extractor"] = ["MelSpectrogram", "MFCC", "Chroma"]
model = train_model(hparams)

In [ ]:
evaluate_model(model)

### MelSpectrogram, MFCC and Chroma with Pretrained Weights
The combination is a stack of MelSpectrogram, MFCC and Chroma features, resulting in a 3-channel feature, with imagenet pretrained weights (IMAGENET1K_V1).

In [ ]:
hparams["experiment"] = "densenet121-classes=8-equalizer-sample_limit=1000-pretrained-Stack_MelSpectrogram_MFCC_Chroma"
hparams["dataset"]["feature_extractor"] = ["MelSpectrogram", "MFCC", "Chroma"]
hparams["model"]["pretrained"] = "IMAGENET1K_V1"
model = train_model(hparams)

In [ ]:
evaluate_model(model)

In [ ]:
%tensorboard --logdir {LOGS_FOLDER}